In [1]:
# basic libraries for data handling
import numpy as np
import pandas as pd

In [110]:
# loading the imdb reviews csv (keep the file in the same folder as this notebook)
df = pd.read_csv(r"IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### To Lower Case

In [111]:
# lowercasing so 'Good' and 'good' are not treated as different words
df['review'] = df['review'].str.lower()

In [112]:
# checking first review to confirm lowercase worked
df['review'][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

### Removing Html tags from reviews 

In [113]:
# review 1 still has <br /> tags in it, need to remove them
df['review'][1]

'a wonderful little production. <br /><br />the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. <br /><br />the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well d

In [114]:
# regex <.*?> matches anything between < and > (non-greedy so it stops at the first >)
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r"",text)

In [115]:
# applying on the whole column, then checking the same review again
df['review'] = df['review'].apply(remove_html_tags)
df['review'][1]

'a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well done.'

### Removing URLs

In [116]:
# some dummy texts to test the url removal function
text_one = "Check out our website at https://www.example.com for more details. You can also visit http://blog.example.org/posts/123."

text_two = "For customer support, go to https://help-center.net. Follow updates at https://twitter.com/example."

text_three = "Download the report here: www.download-link.com/report.pdf or access the portal directly via https://subdomain.portal.io/login."

text_four = "Contact us at info@example.com or browse the catalog at http://shop.example-store.co.uk/category/shoes."

In [117]:
# pattern covers http/https links, www links and email ids
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+|\S+@\S+')
    return pattern.sub(r"",text)

In [118]:
# testing on all 4 sample texts
print(remove_url(text_one))
print(remove_url(text_two))
print(remove_url(text_three))
print(remove_url(text_four))

Check out our website at  for more details. You can also visit 
For customer support, go to  Follow updates at 
Download the report here:  or access the portal directly via 
Contact us at  or browse the catalog at 


### Removing Punctuations

In [119]:
# time is imported to check how long punctuation removal takes
import string, time
string.punctuation  # List of English Punctuations in python

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [120]:
# saving punctuation chars in a variable so we dont call string.punctuation again and again
exclude = string.punctuation

In [121]:
# going through each punctuation char and replacing it with empty string (slow on big data)
def remove_punc(text):
    for char in exclude:
        text = text.replace(char,"")
    return text

In [122]:
# small test string with punctuations
text = "#String! with, punctuations?"

In [123]:
# checking the function on test string
remove_punc(text)

'String with punctuations'

In [124]:
# noting the time taken, this loop way is slow on 50k reviews
start = time.time()
df['review'] = df['review'].apply(remove_punc)
end = time.time() - start

In [125]:
# time taken in seconds
print(end)

1.5885601043701172


In [126]:
# checking review 0 after punctuation removal
df['review'][0]

'one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the wordit is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far awayi would say the main appeal of the show is due to the fact that it goes where other shows wouldnt dare forget pretty pictur

### Spelling Correction 

In [131]:
# textblob is used for spelling correction
from textblob import TextBlob

In [135]:
# wrong spellings on purpose to see if correct() fixes them
incorrect_text = "ceertain condetionns durinng seveal ggeneratons aare modefeid in the samee maner"
textBlb = TextBlob(incorrect_text)
textBlb.correct().string

'certain conditions during several generations are modified in the same manner'

### Removing Stop Words

In [142]:
# nltk already has a stopwords list (download once using nltk.download('stopwords'))
from nltk.corpus import stopwords

In [143]:
# list of english stopwords
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [13]:
# without any language it gives stopwords of all languages together
len(stopwords.words())

11297

In [148]:
# only english stopwords count
len(stopwords.words('english'))

198

In [152]:
# using set because lookup is faster than list
stp_wrds = set(stopwords.words('english'))  

def remove_stop_words(text):
    new_text = []
    for word in text.split():
        if word.lower() not in stp_wrds:  
            new_text.append(word)
            
    return " ".join(new_text)               

In [153]:
# removing stopwords from all reviews
df['review'] = df['review'].apply(remove_stop_words)

In [154]:
# checking review 2 after stopword removal
df['review'][2]

'thought wonderful way spend time hot summer weekend sitting air conditioned theater watching lighthearted comedy plot simplistic dialogue witty characters likable even well bread suspected serial killer may disappointed realize match point 2 risk addiction thought proof woody allen still fully control style many us grown lovethis id laughed one woodys comedies years dare say decade ive never impressed scarlet johanson managed tone sexy image jumped right average spirited young womanthis may crown jewel career wittier devil wears prada interesting superman great comedy go see friends'

In [21]:
# sample speech to practice stopword removal + stemming
speech_str = """I have three visions for India. In 3000 years of our history people from all over the world have 
come and invaded us, captured our lands, conquered our minds. From Alexander onwards the Greeks, the Turks, the Moguls,
the Portuguese, the British, the French, the Dutch, all of them came and looted us, took over what was ours. 
Yet we have not done this to any other nation. We have not conquered anyone. We have not grabbed their land, 
their culture and their history and tried to enforce our way of life on them. Why? Because we respect the freedom of others. 
That is why my FIRST VISION is that of FREEDOM. I believe that India got its first vision of this in 1857, when we started the war of Independence. 
It is this freedom that we must protect and nurture and build on. If we are not free, no one will respect us.

We have 10 percent growth rate in most areas. Our poverty levels are falling. Our achievements are being globally recognised today. 
Yet we lack the self-confidence to see ourselves as a developed nation, self-reliant and self-assured. Isn't this incorrect? MY SECOND VISION for India is DEVELOPMENT. 
For fifty years we have been a developing nation. It is time we see ourselves as a developed nation. We are among top five nations in the world in terms of GDP.

I have a THIRD VISION. India must stand up to the world. Because I believe that unless India stands up to the world, no one will respect us.
Only strength respects strength. We must be strong not only as a military power but also as an economic power. Both must go hand-in-hand.
My good fortune was to have worked with three great minds. Dr.Vikram Sarabhai, of the Dept. of Space, Professor Satish Dhawan, who succeeded him and Dr. Brahm Prakash,
father of nuclear material. I was lucky to have worked with all three of them closely and consider this the great opportunity of my life."""

In [29]:
# first removing stopwords, then stemming whatever is left
# (check is case sensitive so capital words like 'I' and 'In' are not removed)
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords

Snowball = SnowballStemmer('english')
stop_words = set(stopwords.words('english'))

speech = speech_str.split()
stp_wrds_Speech = []
stemmed_speech = []


for word in speech:
    if word not in stop_words:
        stp_wrds_Speech.append(word)
for word in stp_wrds_Speech:
    stemmed_speech.append(Snowball.stem(word))

print(stemmed_speech)
print(len(speech_str))
print(len(stemmed_speech))



['i', 'three', 'vision', 'india.', 'in', '3000', 'year', 'histori', 'peopl', 'world', 'come', 'invad', 'us,', 'captur', 'lands,', 'conquer', 'minds.', 'from', 'alexand', 'onward', 'greeks,', 'turks,', 'moguls,', 'portuguese,', 'british,', 'french,', 'dutch,', 'came', 'loot', 'us,', 'took', 'ours.', 'yet', 'done', 'nation.', 'we', 'conquer', 'anyone.', 'we', 'grab', 'land,', 'cultur', 'histori', 'tri', 'enforc', 'way', 'life', 'them.', 'why?', 'becaus', 'respect', 'freedom', 'others.', 'that', 'first', 'vision', 'freedom.', 'i', 'believ', 'india', 'got', 'first', 'vision', '1857,', 'start', 'war', 'independence.', 'it', 'freedom', 'must', 'protect', 'nurtur', 'build', 'on.', 'if', 'free,', 'one', 'respect', 'us.', 'we', '10', 'percent', 'growth', 'rate', 'areas.', 'our', 'poverti', 'level', 'falling.', 'our', 'achiev', 'global', 'recognis', 'today.', 'yet', 'lack', 'self-confid', 'see', 'develop', 'nation,', 'self-reli', 'self-assured.', "isn't", 'incorrect?', 'my', 'second', 'vision', 

### Replacing/Decoding the Emojis

In [158]:
# emoji library to handle emojis in text
import emoji

In [159]:
# sample lines with emojis
line_one = "I love learning Python! 🐍 This text cleaning pipeline is amazing 🔥🚀"
line_two = "Customer support was terrible 😡, but the product delivery was fast 📦✨"

In [162]:
# demojize turns emoji into text like :fire: and then remove_punc removes the colons
print(remove_punc(emoji.demojize(line_one)))
print(remove_punc(emoji.demojize(line_two)))

I love learning Python snake This text cleaning pipeline is amazing firerocket
Customer support was terrible enragedface but the product delivery was fast packagesparkles
